In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import roc_curve, roc_auc_score
#Load estimation dataset
mort_est_df = pd.read_csv('mort_for_estimation.csv', header=0, parse_dates=['ORIG_DATE1', 'DATE_NOW'])
mort_est_df.describe()
mort_pred_df = pd.read_csv('mort_for_prediction.csv', header=0, parse_dates=['ORIG_DATE1', 'DATE_NOW'])

In [ ]:
#Estimate logit model
X = mort_est_df[['LOAN_AGE', 'OLTV', 'CSCORE_MAX', 'coop_condo_dummy', 'refi_benefit_now', 'refi_benefit_past', 'hpi_growth', 'SATO']]
y = mort_est_df['PREPAY_FLAG']
X = sm.add_constant(X)
logit_model = sm.Logit(y, X)
result = logit_model.fit()
print(result.summary())
y_prob = result.predict(X)

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
#Calculate the in-sample ROC values
fpr, tpr, thresholds = roc_curve(y, y_prob)
roc_auc = roc_auc_score(y, y_prob)

# Plotting
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--') # Diagonal random line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('In-Sample ROC Curve: Prepayment Prediction')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
#Get actual cumulative prepayment rates by DATE_NOW
#Calculate original UPB
start_upb= mort_est_df[['LOAN_ID', 'ORIG_UPB']].drop_duplicates().set_index('LOAN_ID').sum().values[0]

mort_est_df['PREPAID_UPB'] = mort_est_df['PREPAY_FLAG'] * mort_est_df['ORIG_UPB']*mort_est_df['UPB_MULT']
cum_prepay = mort_est_df.groupby('DATE_NOW')['PREPAID_UPB'].sum().cumsum()/start_upb


plt.figure(figsize=(10, 6))
plt.plot(cum_prepay.index, cum_prepay.values)
plt.xlabel('Date')
plt.ylabel('Cumulative Prepayment Rate')
plt.title('Actual Cumulative Prepayment Rate Over Time')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
mort_est_df.head()

In [ ]:
# 1. Select the same features in the same order
X_hypo = mort_pred_df[['LOAN_AGE', 'OLTV', 'CSCORE_MAX', 'coop_condo_dummy', 
                       'refi_benefit_now', 'refi_benefit_past', 'hpi_growth', 'SATO']]

# 2. Add the constant (crucial for statsmodels!)
X_hypo = sm.add_constant(X_hypo, has_constant='add')

# 3. Use the 'result' object from your training to predict
y_prob_hypo = result.predict(X_hypo)

In [ ]:
#Next- use the predicted probabilities to get the expected cumulative prepayment rates over time
#This is a start - need to get cumlative survival rate product up to t-1
#Then loan-level orig_upb*upb_factor*(1-cum_survival_rate)*prepay_probability 
# to get expected prepay at time t
#Then aggretate and divide by total orig_upb to get expected cumulative prepay rate at time t

# 1. Calculate the 'stay' probability for each specific month
mort_pred_df['PREDICTED_PREPAY_PROB'] = y_prob_hypo
mort_pred_df['STAY_PROB'] = 1 - mort_pred_df['PREDICTED_PREPAY_PROB']

# 2. Ensure the data is ordered by loan and time
mort_pred_df = mort_pred_df.sort_values(['LOAN_ID', 'LOAN_AGE'])

# 3. Calculate the cumulative product (running product) per loan
# This gives: (1-P_0) * (1-P_1) * ... * (1-P_n)
mort_pred_df['CUM_STAY_PROB'] = mort_pred_df.groupby('LOAN_ID')['STAY_PROB'].cumprod()

# 4. Shift the results to get the product of PREVIOUS months only
mort_pred_df['LAG_PROD_SURVIVAL'] = (
    mort_pred_df.groupby('LOAN_ID')['CUM_STAY_PROB']
    .shift(1, fill_value=1.0)
)
mort_pred_df.drop(columns=['STAY_PROB', 'CUM_STAY_PROB'], inplace=True)


In [ ]:
mort_pred_df['EXPECTED_PREPAID_UPB'] = mort_pred_df['ORIG_UPB'] * mort_pred_df['UPB_MULT']  * (1 - mort_pred_df['LAG_PROD_SURVIVAL']) * mort_pred_df['PREDICTED_PREPAY_PROB']
agg_expected_prepay = mort_pred_df.groupby('DATE_NOW')['EXPECTED_PREPAID_UPB'].sum().cumsum()/start_upb

plt.figure(figsize=(10, 6))
plt.plot(agg_expected_prepay.index, agg_expected_prepay.values)
plt.xlabel('Date')
plt.ylabel('Cumulative Prepayment Rate')
plt.title('Expected Cumulative Prepayment Rate Over Time')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
mort_pred_df.describe()